<a href="https://colab.research.google.com/github/xandercook172-del/wre-bookings-dashboard/blob/Colab/3rdPartyResearch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
from google.colab import files
import io

uploaded = files.upload()

# Get the filename
filename = list(uploaded.keys())[0]
print(f"\nLoading {filename}...")

# Load the CSV file
df = pd.read_csv(filename)

print(f"✅ Loaded {len(df):,} rows")
print(f"Columns: {df.columns.tolist()}")

# Define Keywords

# Homeowner keywords
HOMEOWNER_KEYWORDS = [
    'homeowner', 'home owner', 'property owner', 'real estate owner',
    'single family', 'residential property', 'housing', 'property value',
    'home value', 'house', 'dwelling', 'own home', 'owns home', 'property type'
]

# Septic keywords
SEPTIC_KEYWORDS = [
    'septic', 'septic tank', 'septic system', 'septic service',
    'sewer', 'wastewater', 'well water', 'well',
    'rural utilities', 'on-site waste', 'septic clean'
]

# REGIONAL INTENT KEYWORDS

# New England - Higher spenders, home expansion/renovation, vehicle service
NEW_ENGLAND_KEYWORDS = [
    'big spend', 'high spend', 'super spend', 'affluent', 'net worth',
    'renovation', 'remodel', 'home improvement', 'home expansion', 'hardware store',
    'vehicle repair', 'vehicle service', 'auto service', 'multi-car', 'multicar',
    'furniture purchase', 'furniture', 'frequent purchaser'
]

# Mid-Atlantic - Recently moved, yard care, wells, planned improvements
MID_ATLANTIC_KEYWORDS = [
    'new mover', 'recently moved', 'recent purchase', 'new home',
    'lawn care', 'garden', 'landscaping', 'yard', 'outdoor',
    'well water', 'well', 'rural',
    'planned improvement', 'plans remodel', 'home entertainment', 'planned'
]

# Mid-South - Large families, cooking, property value mid-high, appliances
MID_SOUTH_KEYWORDS = [
    'children', 'family', 'household size', 'large household', 'kids',
    'cooking', 'kitchen', 'culinary', 'food prep', 'recipes',
    'property value', 'home value', 'real estate value',
    'appliance', 'home depot', 'home improvement store', 'gardening'
]

# South - Second homes, seasonal, HVAC, household size, cooking
SOUTH_KEYWORDS = [
    'second home', 'vacation home', 'seasonal', 'snowbird',
    'household size', 'occupants',
    'hvac', 'air conditioning', 'heating', 'climate control',
    'homeowner insurance', 'state farm', 'insurance',
    'cooking', 'kitchen', 'dining', 'household goods'
]

# Create Filter Functions

def contains_keywords(text, keywords):
    """Check if text contains any of the keywords (case-insensitive)"""
    if pd.isna(text):
        return False
    text_lower = str(text).lower()
    return any(keyword.lower() in text_lower for keyword in keywords)

def has_homeowner_indicator(row):
    """Check if audience has homeowner indicator"""
    combined = f"{row['name']} {row['description']}"
    return contains_keywords(combined, HOMEOWNER_KEYWORDS)

def has_septic_indicator(row):
    """Check if audience has septic indicator"""
    combined = f"{row['name']} {row['description']}"
    return contains_keywords(combined, SEPTIC_KEYWORDS)

def get_regional_match(row):
    """Determine which region(s) this audience matches"""
    combined = f"{row['name']} {row['description']}"

    regions = []

    if contains_keywords(combined, NEW_ENGLAND_KEYWORDS):
        regions.append('New England')
    if contains_keywords(combined, MID_ATLANTIC_KEYWORDS):
        regions.append('Mid-Atlantic')
    if contains_keywords(combined, MID_SOUTH_KEYWORDS):
        regions.append('Mid-South')
    if contains_keywords(combined, SOUTH_KEYWORDS):
        regions.append('South')

    return regions if regions else ['General']

# Apply Filters

print("\n" + "="*60)
print("FILTERING DATA...")
print("="*60)

# Convert reach to numeric
df['reach'] = pd.to_numeric(df['reach'], errors='coerce')

# Filter for reach > 10,000
df = df[df['reach'] > 10000].copy()
print(f"✓ After reach filter (>10,000): {len(df):,} rows")

# Add indicators (with progress tracking)
print("✓ Checking homeowner indicators...")
df['has_homeowner'] = df.apply(has_homeowner_indicator, axis=1)
homeowner_count = df['has_homeowner'].sum()
print(f"  Found {homeowner_count:,} with homeowner indicators")

print("✓ Checking septic indicators...")
df['has_septic'] = df.apply(has_septic_indicator, axis=1)
septic_count = df['has_septic'].sum()
print(f"  Found {septic_count:,} with septic indicators")

print("✓ Matching to regions...")
df['regions'] = df.apply(get_regional_match, axis=1)

# Keep only homeowners
df = df[df['has_homeowner']].copy()
print(f"✓ With homeowner indicator: {len(df):,} rows")

# Create Audience Personas

def categorize_audience(row, region):
    """Categorize audience by persona based on region"""
    combined = f"{row['name']} {row['description']}".lower()
    has_septic = row['has_septic']

    if region == 'New England':
        if has_septic:
            return 'Septic-Ready Homeowners'
        elif any(kw in combined for kw in ['renovation', 'remodel', 'hardware', 'home improvement', 'expansion']):
            return 'Renovation-Driven Homeowners'
        else:
            return 'High-Spending Homeowners'

    elif region == 'Mid-Atlantic':
        if has_septic or 'septic service' in combined:
            return 'Active Septic Service Seekers'
        elif any(kw in combined for kw in ['planned', 'improvement', 'remodel']):
            return 'Planned Home Improvers'
        else:
            return 'Yard-Conscious Homeowners'

    elif region == 'Mid-South':
        if has_septic:
            return 'Confirmed Septic Property Owners'
        elif any(kw in combined for kw in ['appliance', 'kitchen', 'cooking']):
            return 'High-Usage Appliance Homes'
        else:
            return 'Family-Oriented Homeowners'

    elif region == 'South':
        if has_septic:
            return 'Known Septic Homeowners'
        elif any(kw in combined for kw in ['hvac', 'insurance', 'maintenance']):
            return 'Preventative Maintenance Homeowners'
        else:
            return 'Seasonal/High-Usage Homeowners'

    return 'General Homeowner'

# Process Each Region and Get Top 5

print("\n" + "="*60)
print("SELECTING TOP AUDIENCES BY REGION...")
print("="*60)

all_results = []

for region in ['New England', 'Mid-Atlantic', 'Mid-South', 'South']:
    print(f"\n{region}:")

    # Filter for this region
    region_df = df[df['regions'].apply(lambda x: region in x)].copy()

    if len(region_df) == 0:
        print(f"  ⚠️  No candidates found")
        continue

    print(f"  Candidates: {len(region_df):,}")

    # Add persona category
    region_df['persona'] = region_df.apply(lambda row: categorize_audience(row, region), axis=1)

    # Get unique personas for this region
    personas = region_df['persona'].unique()

    for persona in personas:
        persona_df = region_df[region_df['persona'] == persona].copy()

        # Prioritize septic audiences
        septic_df = persona_df[persona_df['has_septic']]
        non_septic_df = persona_df[~persona_df['has_septic']]

        # Get top by reach
        top_septic = septic_df.nlargest(3, 'reach') if len(septic_df) > 0 else pd.DataFrame()

        # Fill remaining slots with non-septic
        remaining_slots = 5 - len(top_septic)
        top_non_septic = non_septic_df.nlargest(remaining_slots, 'reach') if remaining_slots > 0 else pd.DataFrame()

        # Combine
        top_for_persona = pd.concat([top_septic, top_non_septic]).head(5)

        # Add region column
        for idx, row in top_for_persona.iterrows():
            result_row = row.copy()
            result_row['region'] = region
            all_results.append(result_row)

        print(f"    • {persona}: {len(top_for_persona)} audiences")

# Create Final DataFrame

result_df = pd.DataFrame(all_results)

# Remove duplicates
result_df = result_df.drop_duplicates(subset=['name', 'provider_source', 'region'])

print(f"\n{'='*60}")
print(f"✅ FINAL RESULTS: {len(result_df)} audiences selected")
print(f"{'='*60}")

# Export

export_df = result_df[['region', 'persona', 'name', 'description', 'provider_source', 'reach', 'cost', 'has_septic', 'country', 'currency']].copy()

# Sort by region, persona, and reach
export_df = export_df.sort_values(['region', 'persona', 'reach'], ascending=[True, True, False])

# Export to CSV

output_filename = 'Filtered_Top_Audiences_By_Region.csv'
export_df.to_csv(output_filename, index=False)

print(f"\n✅ Exported to: {output_filename}")

# Download the file
files.download(output_filename)

# Summary Statistics

print("\n" + "="*60)
print("SUMMARY BY REGION")
print("="*60)

for region in ['New England', 'Mid-Atlantic', 'Mid-South', 'South']:
    region_data = export_df[export_df['region'] == region]
    if len(region_data) == 0:
        continue
    print(f"\n{region}:")
    print(f"  Total audiences: {len(region_data)}")
    print(f"  With septic indicator: {region_data['has_septic'].sum()}")
    print(f"  Total reach: {region_data['reach'].sum():,.0f}")
    print(f"  Average reach: {region_data['reach'].mean():,.0f}")
    print(f"  Personas:")
    for persona in region_data['persona'].unique():
        count = len(region_data[region_data['persona'] == persona])
        avg_reach = region_data[region_data['persona'] == persona]['reach'].mean()
        print(f"    • {persona}: {count} audiences (avg reach: {avg_reach:,.0f})")

print("\n" + "="*60)
print("TOP 10 PROVIDERS BY TOTAL REACH")
print("="*60)

top_providers = export_df.groupby('provider_source')['reach'].sum().sort_values(ascending=False).head(10)
for provider, total_reach in top_providers.items():
    count = len(export_df[export_df['provider_source'] == provider])
    print(f"  {provider}: {total_reach:,.0f} total reach ({count} audiences)")

print("\n" + "="*60)
print("SAMPLE OF TOP RESULTS")
print("="*60)
print(export_df.head(10)[['region', 'persona', 'provider_source', 'reach']].to_string(index=False))

print("\n✅ COMPLETE!")

Please upload your CSV file...


Saving REQ-257771.csv to REQ-257771 (1).csv

Loading REQ-257771 (1).csv...
✅ Loaded 724,694 rows
Columns: ['name', 'description', 'cost', 'provider_source', 'reach', 'country', 'currency']

FILTERING DATA...
✓ After reach filter (>10,000): 561,042 rows
✓ Checking homeowner indicators...
  Found 69,281 with homeowner indicators
✓ Checking septic indicators...
  Found 13,666 with septic indicators
✓ Matching to regions...
✓ With homeowner indicator: 69,281 rows

SELECTING TOP AUDIENCES BY REGION...

New England:
  Candidates: 2,433
    • High-Spending Homeowners: 5 audiences
    • Renovation-Driven Homeowners: 5 audiences
    • Septic-Ready Homeowners: 3 audiences

Mid-Atlantic:
  Candidates: 3,368
    • Yard-Conscious Homeowners: 5 audiences
    • Active Septic Service Seekers: 3 audiences
    • Planned Home Improvers: 5 audiences

Mid-South:
  Candidates: 6,307
    • Family-Oriented Homeowners: 5 audiences
    • High-Usage Appliance Homes: 5 audiences
    • Confirmed Septic Property Ow

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


SUMMARY BY REGION

New England:
  Total audiences: 13
  With septic indicator: 3
  Total reach: 18,387,323,512
  Average reach: 1,414,409,501
  Personas:
    • High-Spending Homeowners: 5 audiences (avg reach: 2,147,483,647)
    • Renovation-Driven Homeowners: 5 audiences (avg reach: 1,342,516,729)
    • Septic-Ready Homeowners: 3 audiences (avg reach: 312,440,543)

Mid-Atlantic:
  Total audiences: 13
  With septic indicator: 3
  Total reach: 13,911,700,000
  Average reach: 1,070,130,769
  Personas:
    • Active Septic Service Seekers: 3 audiences (avg reach: 1,253,266,667)
    • Planned Home Improvers: 5 audiences (avg reach: 825,180,000)
    • Yard-Conscious Homeowners: 5 audiences (avg reach: 1,205,200,000)

Mid-South:
  Total audiences: 13
  With septic indicator: 3
  Total reach: 19,717,722,734
  Average reach: 1,516,747,903
  Personas:
    • Confirmed Septic Property Owners: 3 audiences (avg reach: 1,155,400,000)
    • Family-Oriented Homeowners: 5 audiences (avg reach: 1,664,39